# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one page (one content item), using the anonymized starter dataset (content_refresh_anonymized.csv, 30,000 pages with 90-day signals). The warehouse daily table (fact_content_daily_performance) is the planned next step, with a mid-panel development month and a sealed final month. All checks below were run on the starter file.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


**Features:**
- `impressions_90d` - real search visibility signal
- `clicks_90d` - real engagement signal
- `avg_position` - real ranking signal
- `word_count` - real content signal
- `days_since_last_update` - real freshness signal

**Label:**
- `trend_direction == "down"` - the proxy target I am trying to predict

**Context:**
- `content_hash_id`, `client_hash_id` - used to join and group tables, not fed into the model
- `report_date` - used to define my time window

**Excluded:**
- `health_score`, `priority_score`, `action_type` - these are FlyRank's own product decisions, not raw signals. Using them would let the model copy the existing rule instead of learning from real data.
- Any data from outside March 2026 - excluded to avoid leakage, since my feature window is fixed to one month.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
import pandas as pd 

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df.head(2)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7


In [12]:
dupes = df['content_id'].duplicated().sum()
print("Duplicate content_id rows:", dupes)

print(df.isna().sum())

print("Age range (days):", df['content_age_days'].min(), "to", df['content_age_days'].max())

Duplicate content_id rows: 0
content_id                    0
client_id                     0
search_volume              2468
competition                2468
competition_level          2610
cpc                        2468
content_type                  0
main_intent                2374
word_count                 7699
char_count                 7699
provider_used             21438
model_used                 5733
impressions_90d               0
clicks_90d                    0
pageviews_90d                 0
sessions_90d                  0
users_90d                     0
engaged_sessions_90d          0
ai_sessions_90d               0
scroll_events_90d             0
days_with_impressions         0
days_with_sessions            0
impressions_last_30d          0
clicks_last_30d               0
sessions_last_30d             0
impressions_prev_30d          0
clicks_prev_30d               0
sessions_prev_30d             0
content_age_days              0
age_tier                      0
age_tier_or

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice has real limits. The history is unbalanced — different clients have different amounts of tracked history, so patterns from clients with long histories may not generalize to newer ones. Some rows only have search data (ga4_data_available = FALSE), meaning early rows for a client can look like "no traffic" when it's really "no tracking yet." March 2026 is also just one month, so this slice can't capture seasonal patterns that show up across a full year. Finally, because my feature window and target window must stay separate to avoid leakage, this slice alone can't yet prove anything about future outcomes, only about the current month.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.